# EDA 6.004: Linear regression residuals for `votes_helpful`

This notebook mirrors the data and normalization choices from `eda_006_normalization_003.ipynb`,
and focuses on simple linear regression baselines for `votes_helpful` (raw vs `log1p`) and
their residuals vs fitted plots.

## Imports and data load

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from steam_review_ml.data.preprocess import (
    filter_reviews,
    load_raw_reviews,
    select_features,
)


def add_review_word_count(temp: pd.DataFrame) -> pd.DataFrame:
    temp["review_word_count"] = temp["review"].fillna("").apply(lambda x: len(str(x).split()))
    return temp


RAW_PATH = Path("../../data/raw/steam_reviews_full.csv")
df_raw = load_raw_reviews(RAW_PATH, nrows=1_000_000)
df_clean = filter_reviews(df_raw)
df = select_features(df_clean)
df = add_review_word_count(df)
print("Shape:", df.shape)

## Column definitions (numeric features to normalize

In [ ]:
CAP_THEN_LOG_COLS = ["votes_helpful", "author.playtime_last_two_weeks"]
LOG_ONLY_COLS = [
    "votes_funny",
    "author.playtime_at_review",
    "author.num_games_owned",
    "author.num_reviews",
    "review_word_count",
]
NORM_COLS = CAP_THEN_LOG_COLS + LOG_ONLY_COLS
NORM_COLS

## TensorFlow normalization (same as 003)

Cap-then-log vs log-only using the `SteamNormalizer` layer, then merge normalized
columns back into the full dataframe as `df_tf`.

In [ ]:
import tensorflow as tf

CAP_P = 0.99
caps = {col: df[col].quantile(CAP_P) for col in CAP_THEN_LOG_COLS}
print("Caps:", caps)


class SteamNormalizer(tf.keras.layers.Layer):
    def __init__(self, caps, cap_then_log_cols, log_only_cols, **kwargs):
        super().__init__(**kwargs)
        self.caps = caps
        self.cap_then_log_cols = cap_then_log_cols
        self.log_only_cols = log_only_cols

    def call(self, inputs):
        x = {k: tf.convert_to_tensor(v) for k, v in inputs.items()}
        for col in self.cap_then_log_cols:
            c = tf.cast(self.caps[col], x[col].dtype)
            x[col] = tf.math.log1p(tf.minimum(x[col], c))
        for col in self.log_only_cols:
            x[col] = tf.math.log1p(x[col])
        return x


norm_layer = SteamNormalizer(
    caps=caps,
    cap_then_log_cols=CAP_THEN_LOG_COLS,
    log_only_cols=LOG_ONLY_COLS,
)

# Build a batch only for the numeric columns we want to normalize
batch = {
    col: tf.constant(df[col].to_numpy().reshape(-1, 1), dtype=tf.float32)
    for col in NORM_COLS
}

normed = norm_layer(batch)

# Start from the original df and overwrite the normalized columns
df_tf = df.copy()
for col in NORM_COLS:
    df_tf[col] = normed[col].numpy().reshape(-1)

df_tf.describe()

## Residuals vs fitted: raw vs log target

Fit two simple linear regression baselines using the normalized features:

1. Predict **raw** `votes_helpful`.
2. Predict **log1p(votes_helpful)`**, then back-transform predictions to raw space via `expm1`.

We then compare residuals vs fitted values (in raw units) to visualize variance patterns.

In [ ]:
from sklearn.linear_model import LinearRegression

# Use normalized features for a simple linear baseline
X = df_tf[NORM_COLS].fillna(0.0).to_numpy()

# Model 1: raw votes_helpful
y_raw = df["votes_helpful"].to_numpy().astype(float)
lin_raw = LinearRegression()
lin_raw.fit(X, y_raw)
yhat_raw = lin_raw.predict(X)
resid_raw = y_raw - yhat_raw

# Model 2: log1p(votes_helpful)
y_log = np.log1p(y_raw)
lin_log = LinearRegression()
lin_log.fit(X, y_log)
yhat_log = lin_log.predict(X)
# Back-transform predictions to raw scale for residuals
yhat_log_raw = np.expm1(yhat_log)
resid_log_raw = y_raw - yhat_log_raw

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Residuals vs fitted (raw target) with symlog
axes[0].scatter(yhat_raw, resid_raw, alpha=0.1, s=2)
axes[0].axhline(0, color="red", linewidth=1)
axes[0].set_xlabel("Fitted values (raw votes_helpful)")
axes[0].set_ylabel("Residuals (raw)")
axes[0].set_title("Residuals vs fitted (raw target, symlog Y)")
axes[0].set_yscale("symlog", linthresh=1.0)

# Residuals vs fitted (log target, back-transformed) with symlog
axes[1].scatter(yhat_log_raw, resid_log_raw, alpha=0.1, s=2)
axes[1].axhline(0, color="red", linewidth=1)
axes[1].set_xlabel("Fitted values (back-transformed from log1p)")
axes[1].set_ylabel("Residuals (raw)")
axes[1].set_title("Residuals vs fitted (log1p target, back-transformed, symlog Y)")
axes[1].set_yscale("symlog", linthresh=1.0)

plt.tight_layout()
plt.show()